# Customer Churn Definition & Target Engineering
### Olist Brazilian E-Commerce Dataset

In this notebook, we define and engineer the binary classification target **`is_churned`**:
- **Cutoff Date**: Maximum purchase timestamp in the dataset minus 6 months (`max_date - 6 months`).
- **Target Logic**:
  - `is_churned = 1` (Churned): Customer has made **no purchase** in the last 6 months (`last_purchase < cutoff_date`).
  - `is_churned = 0` (Retained / Active): Customer made at least one purchase within the last 6 months (`last_purchase >= cutoff_date`).
- Merge `is_churned` with `rfm_scaled` into a modeling DataFrame and evaluate **Class Imbalance**.

In [1]:
import os
import pandas as pd
import numpy as np

# 1. Load Processed Datasets
master_path = os.path.join("..", "Data", "Processed", "df_master.csv") if os.path.exists(os.path.join("..", "Data", "Processed", "df_master.csv")) else os.path.join("Data", "Processed", "df_master.csv")
scaled_path = os.path.join("..", "Data", "Processed", "rfm_scaled.csv") if os.path.exists(os.path.join("..", "Data", "Processed", "rfm_scaled.csv")) else os.path.join("Data", "Processed", "rfm_scaled.csv")

df_master = pd.read_csv(master_path)
df_master["order_purchase_timestamp"] = pd.to_datetime(df_master["order_purchase_timestamp"])

rfm_scaled = pd.read_csv(scaled_path, index_col="customer_unique_id")

print(f"Loaded df_master:  {df_master.shape}")
print(f"Loaded rfm_scaled: {rfm_scaled.shape}")

Loaded df_master:  (115035, 22)
Loaded rfm_scaled: (93357, 3)


## 2. Define 6-Month Inactivity Cutoff & Calculate `is_churned`

In [2]:
# Determine max dataset timestamp & 6-month cutoff date
max_date = df_master["order_purchase_timestamp"].max()
cutoff_date = max_date - pd.DateOffset(months=6)

print(f"Dataset Max Purchase Date: {max_date}")
print(f"6-Month Cutoff Date:       {cutoff_date}")

# Find last purchase date per customer_unique_id
last_purchase_per_customer = df_master.groupby("customer_unique_id")["order_purchase_timestamp"].max()

# Create binary target: 1 = Churned (< cutoff_date), 0 = Retained (>= cutoff_date)
churn_target = (last_purchase_per_customer < cutoff_date).astype(int).rename("is_churned")

# Merge target variable with scaled RFM features
df_model = rfm_scaled.join(churn_target)

print(f"\nMerged Modeling DataFrame shape: {df_model.shape}")
display(df_model.head())

Dataset Max Purchase Date: 2018-08-29 15:00:37
6-Month Cutoff Date:       2018-02-28 15:00:37

Merged Modeling DataFrame shape: (93357, 4)


,Recency,Frequency,Monetary,is_churned
customer_unique_id,,,,
0000366f3b9a7992bf8c76cfdf3221e2,-0.459456,-0.172192,0.169334,0
0000b849f77a49e4a4ce2b2a4ca5be3f,-0.431651,-0.172192,-1.637174,0
0000f46a3911fa3c0805444483337064,1.189395,-0.172192,-0.375394,1
0000f6ccb0745a6a4b88665a16c9f078,0.648133,-0.172192,-1.120386,1
0004aac84e0df4da2b147fca70cf8255,0.534022,-0.172192,0.527429,1


## 3. Class Imbalance Analysis

In [3]:
counts = df_model["is_churned"].value_counts()
percentages = df_model["is_churned"].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    "Class Label": counts.index.map({1: "1 (Churned)", 0: "0 (Retained / Active)"}),
    "Customer Count": counts.values,
    "Proportion (%)": percentages.values
})

print("=== CHURN CLASS DISTRIBUTION ===")
display(class_summary)

# Save dataset for modeling
processed_dir = os.path.join("..", "Data", "Processed") if os.path.exists(os.path.join("..", "Data", "Processed")) else os.path.join("Data", "Processed")
model_csv_path = os.path.join(processed_dir, "df_churn_model.csv")
df_model.to_csv(model_csv_path)
print(f"\nSaved modeling dataset to: {model_csv_path}")

=== CHURN CLASS DISTRIBUTION ===


,Class Label,Customer Count,Proportion (%)
0,1 (Churned),54712,58.605139
1,0 (Retained / Active),38645,41.394861



Saved modeling dataset to: ..\Data\Processed\df_churn_model.csv
